# Automotive Supply Chain Analysis with yFiles <a target="_blank" href="https://colab.research.google.com/github/yWorks/yfiles-jupyter-graphs/blob/main/examples/showcases/supply_chain.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how to explore and visualize a complex automotive supply chain using `yfiles-jupyter-graphs`. We will transform raw CSV data into an interactive graph that highlights tiers, transport modes, and logistics volumes.

### Learning Goals
- Loading supply chain data from CSV.
- Creating an interactive graph visualization.
- Applying data-driven styles to nodes and edges.
- Using hierarchical layouts to reveal supply chain structure.

### Before using the widget, make sure to install the required packages

Ensure you have the necessary packages installed by running the following command:
- ```%pip install yfiles_jupyter_graphs pandas --quiet```

In [ ]:
%pip install yfiles_jupyter_graphs pandas --quiet

You can also open this notebook in Google Colab when Google Colab's custom widget manager is enabled:

In [ ]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

<a target="_blank" href="https://colab.research.google.com/github/yWorks/yfiles-jupyter-graphs/blob/main/examples/showcases/supply_chain.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup & Data Loading

First, we import the necessary libraries and load our supply chain dataset.

In [ ]:
import pandas as pd
from yfiles_jupyter_graphs import GraphWidget, Layout, NodeStyle, NodeShape, EdgeStyle, DashStyle, LabelStyle

# Load the data
url = 'https://raw.githubusercontent.com/yWorks/yfiles-jupyter-graphs/refs/heads/main/examples/showcases/resources/supply_chain_data.csv'
df = pd.read_csv(url)

# Basic exploration
print(f"Dataset shape: {df.shape}")
display(df.head())

print("\nTransport mode distribution:")
print(df['transport_mode'].value_counts())

print("\nQuantity statistics:")
print(df['quantity'].describe())

## Step 1: Basic Visualization

We can import the Pandas DataFrame directly into `GraphWidget`. By default, it uses the `source` and `target` columns to define edges. We'll start with the default force-directed layout.

In [ ]:
w = GraphWidget()
w.import_graph(df)
display(w)

## Step 2: Hierarchical Layout

A supply chain is naturally hierarchical. The `HIERARCHICAL` layout is perfect here as it automatically organizes nodes by their tier level (T3 -> T2 -> T1 -> OEM -> Logistics).

In [ ]:
w.graph_layout = Layout.HIERARCHICAL
display(w)

## Step 3: Advanced Data-Driven Styling & Grouping

To make the visualization truly insightful, we'll apply advanced styles:
1. **Node Grouping**: Group nodes by Tier using `node_parent_group_mapping` to reveal the organizational structure.
2. **Node Colors**: Color-coded by part tier.
3. **Edge Colors**: Color-coded by `transport_mode` (Truck, Ship, Rail, Air).
4. **Edge Thickness**: Scaled by the `quantity` of materials flowing through the connection.
5. **Interactive UI**: Start with the Search panel open to allow quick navigation.

In [ ]:
def get_tier_info(node):
    node_id = str(node['id'])
    if node_id.startswith('T3'): return {"label": "Tier 3 (Raw Materials)", "color": "#DEEBFF"}
    if node_id.startswith('T2'): return {"label": "Tier 2 (Refinement)", "color": "#B3D4FF"}
    if node_id.startswith('T1'): return {"label": "Tier 1 (Sub-systems)", "color": "#4C9AFF"}
    if node_id == 'OEM': return {"label": "OEM (Assembly)", "color": "#FFFAE6"}
    return {"label": "Distribution & Retail", "color": "#EAE6FF"}

def get_node_color(node):
    node_id = str(node['id'])
    if node_id.startswith('T3'): return '#0052CC' 
    if node_id.startswith('T2'): return '#2684FF' 
    if node_id.startswith('T1'): return '#0065E0' 
    if node_id == 'OEM': return '#FF8B00'         
    if 'Logistics' in node_id: return '#36B37E'   
    return '#DFE1E6'

transport_colors = {
    'truck': '#42526E', # Grey
    'ship': '#008DA6',  # Teal
    'rail': '#FF5630',  # Red-Orange
    'air': '#6554C0'    # Purple
}

# Apply advanced mappings
w.node_parent_group_mapping = get_tier_info
w.node_color_mapping = get_node_color
w.node_label_mapping = lambda n: str(n['id']).replace('_', ' ')

w.edge_color_mapping = lambda e: transport_colors.get(e['properties'].get('transport_mode'), '#333333')
w.edge_thickness_factor_mapping = lambda e: e['properties'].get('quantity', 1000) / 1500
w.edge_label_mapping = lambda e: e['properties'].get('transport_mode', '')

display(w)

## Conclusion

By combining `yfiles-jupyter-graphs` with hierarchical layouts, data-driven mappings, and node grouping, we've turned a flat CSV file into a rich, interactive map of the automotive supply chain. This visualization reveals:
- The **hierarchical dependency** from raw ores to dealerships, automatically organized by the layout engine.
- The **organizational tiers** through dynamic node grouping.
- Diverse **logistics modes** and their associated volumes (quantity) through color and thickness mappings.
- A searchable, interactive interface ready for deeper analysis.